# 04 · Orchestrate — 03 Retrying because the verdict said so

**`01-tools/05-gate/01-grounding-check.ipynb` produces a verdict — `grounded: True / False / None`, a score, and the specific quotes that don't check out. Nothing in this repo reads it. It is a report, printed at the end of a pipeline that has already finished. This notebook makes it an input: retrieve, check, and retry *only because the check came back weak*.**

This is the core of the stage. `01-should-i-retrieve.ipynb` decided whether
to call the tool; `02-source-choice.ipynb` decided which source to call
first. Both decisions are made *before* any evidence exists. This one is made
*after* — the first point in this repo where what came back changes what
happens next.

The contrast with `04-retrieve/09-multi-query.ipynb` is exact and worth
holding onto. Multi-query runs every phrasing up front, every time, and
merges. This notebook runs the second phrasing *only because the first one's
result failed a check* — and merges the same way when it does. Same merge,
different reason for the second query existing.

## The cap is not optional

A loop that retries until it is satisfied, driven by a judgment that may
never be satisfied, is an unbounded loop with a paid model inside it. That
is the precise failure `nbio.cost_meter()` exists to catch. Two independent
brakes are used here and both are demonstrated by execution, not asserted in
prose: a hard attempt cap (Step 7, against a checker rigged to never be
satisfied), and the spend ceiling (Step 8, driven until it raises).

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `retrieve` | One phrasing against the course store | `retrieve("mitochondria ATP production")` |
| `merge` | Max score per chunk id across attempts, same rule as `09-multi-query` | `merge(previous, new_hits)` |
| `synthesize` | Deterministic answer stand-in: quotes the top chunk, or over-claims when retrieval is thin | `synthesize(q, results)` |
| `grounding_verdict` | Reduced port of `05-gate`'s deterministic pass: `grounded` + `score` | `grounding_verdict(answer, results)` |
| `is_weak` | The one rule that triggers a retry — `False`, or `None` below the score floor | `is_weak({"grounded": None, "score": 0.5})` |
| `REFORMULATIONS` | The hand-written alternative phrasings a retry draws from | `REFORMULATIONS[0]` |
| `answer_with_retry` | The whole loop: attempt, check, retry if weak, stop at the cap | `answer_with_retry(q, max_attempts=2)` |
| `always_weak_verdict` | A checker rigged never to be satisfied, used to prove the cap holds | Step 7 |

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()

## Step 1 — the store, and the gap between two phrasings of one question

Same construction as `04-retrieve/09-multi-query.ipynb`, kept
self-contained. The stored chunk says "mitochondria" and "oxidative
phosphorylation"; it never says "powerhouse of the cell". The offline hash
embedding has no notion of synonymy, so the informal phrasing scores zero
against a chunk that answers it perfectly.

That zero is the whole setup. It is a retrieval failure caused by the
query, not by the corpus — and the point of this notebook is that the
system finds that out from a *grounding verdict on the answer*, not from
looking at the retrieval score directly.

In [ ]:
import hashlib
import math

RETRIEVAL_CALLS = []


def hash_embed(text: str, dim: int = 384) -> list[float]:
    vec = [0.0] * dim
    for tok in (text or "").lower().split():
        h = int(hashlib.sha256(tok.encode("utf-8")).hexdigest(), 16)
        vec[h % dim] += 1.0 if (h >> 8) & 1 else -1.0
    norm = math.sqrt(sum(v * v for v in vec)) or 1.0
    return [v / norm for v in vec]


def cosine(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a)) or 1.0
    nb = math.sqrt(sum(x * x for x in b)) or 1.0
    return dot / (na * nb)


COURSE_STORE = [
    {"chunk_id": "bio201::c0", "title": "Cellular respiration",
     "text": "Mitochondria perform oxidative phosphorylation, converting nutrients into ATP."},
    {"chunk_id": "bio201::c1", "title": "Photosynthesis",
     "text": "Photosynthesis converts light energy into chemical energy stored in glucose."},
]
for _ch in COURSE_STORE:
    _ch["embedding"] = hash_embed(_ch["text"])


def retrieve(query: str, top_k: int = 2) -> list[dict]:
    RETRIEVAL_CALLS.append(query)
    qvec = hash_embed(query)
    scored = [{"chunk_id": c["chunk_id"], "title": c["title"], "text": c["text"],
               "score": cosine(qvec, c["embedding"])} for c in COURSE_STORE]
    scored.sort(key=lambda x: -x["score"])
    return scored[:top_k]


QUESTION = "what is the powerhouse of the cell"
FORMAL = "What is the role of mitochondria in oxidative phosphorylation?"

print(f"informal phrasing top score: {retrieve(QUESTION)[0]['score']:.3f}")
print(f"formal   phrasing top score: {retrieve(FORMAL)[0]['score']:.3f}")
assert retrieve(QUESTION)[0]["score"] < retrieve(FORMAL)[0]["score"]

## Step 2 — `merge`: the same max-score-per-chunk rule as `09-multi-query`

Ported unchanged from `retrieve_multi_query`'s merge: a chunk seen by more
than one phrasing keeps the *higher* of its scores, so a chunk the first
attempt missed still surfaces at the score of the attempt that found it.
Only the reason for there being a second set of results is different.

In [ ]:
def merge(previous: list[dict], new_hits: list[dict]) -> list[dict]:
    merged = {r["chunk_id"]: r for r in previous}
    for r in new_hits:
        prev = merged.get(r["chunk_id"])
        if prev is None or r["score"] > prev["score"]:
            merged[r["chunk_id"]] = r
    return sorted(merged.values(), key=lambda x: -x["score"])


merged_demo = merge(retrieve(QUESTION), retrieve(FORMAL))
print(f"informal alone : {retrieve(QUESTION)[0]['score']:.3f}")
print(f"merged         : {merged_demo[0]['score']:.3f}")

assert merged_demo[0]["score"] >= retrieve(QUESTION)[0]["score"]
assert len({r["chunk_id"] for r in merged_demo}) == len(merged_demo), "no chunk id appears twice"

## Step 3 — `synthesize`: a stand-in generator that over-claims when retrieval is thin

No model is called here. This is a deterministic stand-in with one
behaviour that matters: when the best retrieved chunk clears
`SUPPORT_FLOOR` it quotes that chunk verbatim, and when nothing clears the
floor it writes a confident sentence that appears in no retrieved
document.

That second branch is not a cheap trick — it is the single most common
real failure mode this repo has instrumentation for. `04-benchmarks`
tracks it as *answered-without-evidence*: thin retrieval, confident prose,
nothing behind it. Reproducing it deterministically is what lets the rest
of this notebook run with no key and still be about something real.

In [ ]:
SUPPORT_FLOOR = 0.15
UNSUPPORTED_SENTENCE = "The retrieved sources state this directly and without qualification."


def synthesize(question: str, results: list[dict]) -> str:
    """Quote the best chunk if retrieval was good enough; otherwise over-claim."""
    if results and results[0]["score"] >= SUPPORT_FLOOR:
        return f'\U0001F4C4 "{results[0]["text"]}"'
    return f'\U0001F4C4 "{UNSUPPORTED_SENTENCE}"'


thin_answer = synthesize(QUESTION, retrieve(QUESTION))
good_answer = synthesize(FORMAL, retrieve(FORMAL))
print("thin retrieval ->", thin_answer)
print("good retrieval ->", good_answer)

assert UNSUPPORTED_SENTENCE in thin_answer
assert "Mitochondria" in good_answer

## Step 4 — `grounding_verdict`: the reduced port of `05-gate`'s deterministic pass

The parts of `01-tools/05-gate/01-grounding-check.ipynb` this loop needs,
and no more: normalise the retrieved context, pull every quoted span out
of the answer, fuzzy-match each one against the context, and report.

The three-valued result is kept exactly as the gate produces it, because
the middle value is the interesting one:

- `False` — a quoted span provably is not in the evidence.
- `None` — nothing was found wrong, but with no LLM judge configured that
  is *not* the same as "definitely grounded". The gate refuses to say
  `True` on the deterministic signal alone, and so does this.
- `True` — only reachable when a judge has confirmed it, which needs a key.

A retry policy therefore cannot key off `grounded is False` alone. It has
to handle `None` too, which is what Step 5 does.

In [ ]:
import re
from difflib import SequenceMatcher

QUOTE_MATCH_THRESHOLD = 0.85
_ESCAPE_RE = re.compile(r"\\[nrt]")


def norm(s: str) -> str:
    return " ".join(_ESCAPE_RE.sub(" ", s or "").replace("\\", "").lower().split())


def context_text(papers: list[dict]) -> str:
    blocks, seen = [], set()
    for p in papers:
        for part in (str(p.get("title") or ""), p.get("text") or ""):
            block = norm(part)
            if block and block not in seen:
                blocks.append(block)
                seen.add(block)
    return "\n".join(blocks)


def quote_supported(quote: str, context_norm: str) -> bool:
    q = norm(quote)
    if len(q) < 12:
        return True  # too short to judge; don't penalize
    if q in context_norm:
        return True
    sm = SequenceMatcher(None, q, context_norm)
    return sm.find_longest_match(0, len(q), 0, len(context_norm)).size >= int(len(q) * QUOTE_MATCH_THRESHOLD)


def grounding_verdict(answer_text: str, papers: list[dict]) -> dict:
    """grounded True/False/None plus a score, on the deterministic signal alone."""
    if not papers or not (answer_text or "").strip():
        return {"grounded": False, "score": 0.0, "unsupported": [], "method": "precondition"}
    ctx = context_text(papers)
    quotes = re.findall(r'"([^"\n]{12,400})"', answer_text)
    unsupported = [q for q in quotes if not quote_supported(q, ctx)]
    if unsupported:
        return {"grounded": False, "score": 0.5, "unsupported": unsupported[:3], "method": "deterministic"}
    return {"grounded": None, "score": 1.0, "unsupported": [], "method": "deterministic"}


v_thin = grounding_verdict(thin_answer, retrieve(QUESTION))
v_good = grounding_verdict(good_answer, retrieve(FORMAL))
print("thin answer verdict:", v_thin)
print("good answer verdict:", v_good)

assert v_thin["grounded"] is False, "an unsupported quote must be caught deterministically"
assert v_good["grounded"] is None, "no problem found, but no judge either — not 'definitely grounded'"
assert v_good["score"] == 1.0

## Step 5 — `is_weak`: the one rule that decides whether a retry happens

Weak means either of two things:

- `grounded is False` — something is provably wrong with the answer.
- `grounded is None` **and** the score is below the floor — nothing was
  proven wrong, but nothing confirmed it either, and the score is not
  reassuring.

`grounded is None` with a *high* score is explicitly not weak. That is the
deterministic pass saying "I found no fabrication and I am not equipped to
say more", and retrying on it would mean retrying on every clean answer
this repo can produce without a key — a loop that never terminates for
the most common case is worse than no loop.

In [ ]:
WEAK_SCORE_FLOOR = 0.7


def is_weak(verdict: dict) -> bool:
    if verdict["grounded"] is False:
        return True
    if verdict["grounded"] is None and verdict["score"] < WEAK_SCORE_FLOOR:
        return True
    return False


cases = [
    ({"grounded": False, "score": 0.5}, True, "provably unsupported"),
    ({"grounded": None, "score": 0.5}, True, "unconfirmed and low-scoring"),
    ({"grounded": None, "score": 1.0}, False, "unconfirmed but nothing wrong found"),
    ({"grounded": True, "score": 0.9}, False, "judge-confirmed"),
]
nbio.table([(str(v["grounded"]), v["score"], is_weak(v), why) for v, _, why in cases],
           ("grounded", "score", "retry?", "reading"))

for verdict, expected, _ in cases:
    assert is_weak(verdict) is expected

## Step 6 — `answer_with_retry`: the loop, with every attempt logged

The whole thing in one function. Each attempt retrieves with the current
query, merges into whatever previous attempts found, synthesizes, and gets
a verdict. The verdict decides whether there is another attempt, and the
log records that decision for every attempt — so a reader can see not just
*that* a retry happened but which verdict caused it, and equally, on the
attempts where no retry happened, why not.

Every attempt is recorded on the meter. Offline there is no paid call to
record, so zero tokens are recorded against the model id — the entry still
counts the attempt, which is the point: `meter.calls` is a count of loop
iterations that reached the generation step, whether or not any of them
cost money. With a key set, the same line carries real usage and the
ceiling is live.

In [ ]:
MODEL_ID = "llama-3.1-8b-instant"

REFORMULATIONS = [
    "What is the role of mitochondria in oxidative phosphorylation?",
    "mitochondria ATP production",
    "cellular respiration energy conversion",
]


def answer_with_retry(question: str, max_attempts: int = 2, meter=None,
                      checker=grounding_verdict, reformulations=REFORMULATIONS) -> dict:
    """Retrieve, check, and retry on a weak verdict — never more than max_attempts times."""
    assert max_attempts >= 1, "at least one attempt, and a finite number of them"
    query, results, log = question, [], []

    for attempt in range(1, max_attempts + 1):
        results = merge(results, retrieve(query))
        answer = synthesize(question, results)
        verdict = checker(answer, results)
        if meter is not None:
            # No paid call on the offline path, so zero tokens — the entry still
            # puts this attempt through the ceiling. With a key set, real usage
            # from the generation call is recorded here instead.
            meter.record(MODEL_ID, 0, 0)

        weak = is_weak(verdict)
        at_cap = attempt >= max_attempts
        next_query = reformulations[attempt - 1] if attempt - 1 < len(reformulations) else None
        if weak and at_cap:
            decision = "weak, but the attempt cap is reached — stopping"
        elif weak and next_query is None:
            decision = "weak, but no reformulation left — stopping"
        elif weak:
            decision = f"weak -> retrying as {next_query!r}"
        else:
            decision = "not weak — accepted"

        log.append({"attempt": attempt, "query": query, "top_score": round(results[0]["score"], 3),
                    "grounded": verdict["grounded"], "score": verdict["score"], "decision": decision})

        if not weak or at_cap or next_query is None:
            return {"answer": answer, "verdict": verdict, "results": results,
                    "attempts": attempt, "stopped_at_cap": weak and at_cap, "log": log}
        query = next_query

    raise AssertionError("unreachable: the loop returns on or before the final attempt")


def show_log(run: dict) -> None:
    nbio.table([(r["attempt"], r["query"][:42], r["top_score"], str(r["grounded"]), r["score"], r["decision"])
                for r in run["log"]],
               ("#", "query", "top score", "grounded", "gscore", "what happened next"))


RETRIEVAL_CALLS.clear()
with nbio.cost_meter(budget_usd=0.50) as meter:
    run = answer_with_retry(QUESTION, max_attempts=2, meter=meter)
    metered_calls = meter.calls
    print(meter.report())

print()
show_log(run)
print(f"\nfinal answer: {run['answer']}")

assert run["attempts"] == 2, "the first attempt's verdict was weak, so a second one happened"
assert run["log"][0]["grounded"] is False
assert run["log"][1]["grounded"] is None and run["log"][1]["score"] == 1.0
assert run["stopped_at_cap"] is False, "it stopped because the verdict improved, not because it ran out of attempts"
assert metered_calls == run["attempts"], "every attempt went through the meter"
assert len(RETRIEVAL_CALLS) == run["attempts"]

## Step 6b — the same question, with a strong first attempt: no retry at all

The loop has to be equally visible when it does *nothing*. Asking the
formal phrasing directly produces a verdict that is not weak on attempt
one, and the log shows a single row — no second retrieval, no second
metered call.

This is the case that separates this notebook from
`04-retrieve/09-multi-query.ipynb`. Multi-query would have run both
phrasings here regardless. This runs one, because one was enough.

In [ ]:
RETRIEVAL_CALLS.clear()
with nbio.cost_meter(budget_usd=0.50) as meter:
    strong_run = answer_with_retry(FORMAL, max_attempts=2, meter=meter)
    strong_metered = meter.calls

show_log(strong_run)

assert strong_run["attempts"] == 1, "a verdict that is not weak must not trigger a retry"
assert len(RETRIEVAL_CALLS) == 1, "and the second retrieval must genuinely not happen"
assert strong_metered == 1
print("\nconfirmed: one attempt, one retrieval, one metered call — the retry was skipped, not merely unlogged")

## Step 7 — proving the cap, against a checker that can never be satisfied

The assertion "it stops after N" is worth nothing unless something has
actually tried to make it not stop. `always_weak_verdict` ignores the
answer and the evidence entirely and returns `grounded: False` every time.
Under it, no reformulation can ever succeed and the *only* thing that can
end the loop is the cap.

It is run twice, at cap 2 and cap 3, against a reformulation pool with
enough entries for either. Both runs stop at exactly their cap, and the
retrieval counter confirms the loop did not sneak in extra lookups. Two
different caps producing two different attempt counts is what shows the
cap is the cause — a single run stopping at 2 could always have been the
data.

In [ ]:
def always_weak_verdict(answer_text: str, papers: list[dict]) -> dict:
    """A grounding check rigged never to be satisfied, so only the cap can stop the loop."""
    return {"grounded": False, "score": 0.0, "unsupported": ["(forced)"], "method": "forced-weak"}


cap_runs = {}
for cap in (2, 3):
    RETRIEVAL_CALLS.clear()
    with nbio.cost_meter(budget_usd=0.50) as meter:
        r = answer_with_retry(QUESTION, max_attempts=cap, meter=meter, checker=always_weak_verdict)
        r["metered_calls"] = meter.calls
    r["retrievals"] = len(RETRIEVAL_CALLS)
    cap_runs[cap] = r
    print(f"--- cap = {cap} ---")
    show_log(r)
    print()

nbio.table([(cap, r["attempts"], r["retrievals"], r["metered_calls"], r["stopped_at_cap"])
            for cap, r in cap_runs.items()],
           ("cap", "attempts made", "retrievals", "metered calls", "stopped at cap"))

for cap, r in cap_runs.items():
    assert r["attempts"] == cap, f"cap {cap} must produce exactly {cap} attempts"
    assert r["retrievals"] == cap, "one retrieval per attempt, no more"
    assert r["metered_calls"] == cap
    assert r["stopped_at_cap"] is True, "it stopped because it ran out of attempts, not because it succeeded"
    assert r["verdict"]["grounded"] is False, "and it stopped still holding a weak verdict"
    assert len(REFORMULATIONS) >= cap - 1, "the pool had spare reformulations, so the pool was not the limit"

assert cap_runs[2]["attempts"] != cap_runs[3]["attempts"], (
    "different caps give different attempt counts — the cap is what stopped it, not the data"
)
print()
print("confirmed: with a verdict that never improves, the loop stops at the cap, still weak, and says so")

## Step 8 — the other brake: driving the spend ceiling until it raises

The attempt cap bounds how many times the loop runs. The spend ceiling
bounds what those runs are allowed to cost, and the two fail
independently — a cap of 2 around a call with a million-token prompt is
still an expensive mistake.

Here the meter is given a deliberately tiny budget and fed the token
counts a real 8B-model attempt would carry, until it raises
`nbio.BudgetExceeded`. Nothing is called and nothing is spent; the point
is that the ceiling is a live mechanism in this loop's path rather than a
decorative `with` block.

In [ ]:
from nbio import BudgetExceeded

TINY_BUDGET = 0.0001
PROMPT_TOKENS, COMPLETION_TOKENS = 1200, 300

stopped_after = None
try:
    with nbio.cost_meter(budget_usd=TINY_BUDGET) as meter:
        for attempt in range(1, 21):  # far more attempts than any cap here allows
            meter.record(MODEL_ID, PROMPT_TOKENS, COMPLETION_TOKENS)
            print(f"  attempt {attempt}: running total ${meter.cost_usd:.6f}")
except BudgetExceeded as exc:
    stopped_after = meter.calls
    print(f"\nBudgetExceeded raised: {exc}")

assert stopped_after is not None, "the ceiling must actually raise, not merely be configured"
assert stopped_after < 20, "and it must raise before the loop would have finished on its own"
print(f"\nconfirmed: the ceiling stopped the run after {stopped_after} metered attempt(s), "
      f"below a ${TINY_BUDGET} budget")

## Step 9 — what the retry actually bought, side by side

One table, three rows: the informal question with no retry allowed
(`max_attempts=1`), the same question with the retry loop, and
`09-multi-query`'s policy of running every phrasing up front regardless.

The retry and the multi-query row reach the same verdict. The difference
is that multi-query paid for the second retrieval on every question,
including the ones the first phrasing already answered — Step 6b's run is
exactly that case, and it is the common one.

In [ ]:
RETRIEVAL_CALLS.clear()
no_retry = answer_with_retry(QUESTION, max_attempts=1)
no_retry_calls = len(RETRIEVAL_CALLS)

RETRIEVAL_CALLS.clear()
with_retry = answer_with_retry(QUESTION, max_attempts=2)
with_retry_calls = len(RETRIEVAL_CALLS)

RETRIEVAL_CALLS.clear()
upfront = merge(retrieve(QUESTION), retrieve(REFORMULATIONS[0]))
upfront_answer = synthesize(QUESTION, upfront)
upfront_verdict = grounding_verdict(upfront_answer, upfront)
upfront_calls = len(RETRIEVAL_CALLS)

nbio.table(
    [("no retry (cap 1)", no_retry_calls, str(no_retry["verdict"]["grounded"]), no_retry["verdict"]["score"]),
     ("retry on weak verdict", with_retry_calls, str(with_retry["verdict"]["grounded"]), with_retry["verdict"]["score"]),
     ("every phrasing up front", upfront_calls, str(upfront_verdict["grounded"]), upfront_verdict["score"])],
    ("policy", "retrievals", "grounded", "score"),
)

assert no_retry["verdict"]["grounded"] is False, "capped at one attempt, the weak answer is what you keep"
assert with_retry["verdict"]["grounded"] is None and with_retry["verdict"]["score"] == 1.0
assert upfront_verdict["grounded"] is with_retry["verdict"]["grounded"], (
    "running both phrasings up front reaches the same verdict as retrying into the second one"
)
assert with_retry_calls == upfront_calls == 2
print()
print("confirmed: on this question the two policies tie; on Step 6b's question the retry policy "
      "makes one retrieval where the up-front policy would make two")

## What did not come across

- **One question, one store, one fabrication.** Everything above runs on a
  two-chunk synthetic store and a generator stand-in built to over-claim
  on cue. It proves the *wiring* — that a verdict is read, that a retry
  happens only when it is weak, that the cap holds under a checker rigged
  to defeat it. It proves nothing about how often a real retry improves a
  real answer. That measurement is
  `04-benchmarks/clinical-retrieval/`, which is separate work.
- **The reformulations are hand-written**, exactly as in
  `04-retrieve/09-multi-query.ipynb`. Generating them with a model is
  query rewriting, a capability this repo does not build anywhere.
- **The `True` branch of `grounded` is never exercised offline.** Reaching
  it needs the LLM judge in `01-tools/05-gate`, which needs a key. The
  loop handles it — `is_weak` returns `False` for it — but no execution
  here demonstrates it.
- **The verdict here is the deterministic pass only.** Fabricated quotes
  and nothing else. The gate's orphan-citation check, bibliographic
  false-positive guard, and judge-merge rules are all in
  `01-tools/05-gate/01-grounding-check.ipynb` and not duplicated here.
- **`meter.calls` counts attempts, not spend, on the offline path.** Zero
  tokens are recorded because zero tokens are used. Step 8 is what shows
  the ceiling doing its actual job.